# Worker Nodes

`PENRSWorker` is the base class for all PENRS agent workers. Each worker fetches a rubric and a financial document, truncates the document to fit the LLM context window, invokes the LLM, and parses the response into a structured result dict. This notebook contains all worker logic and can be run top-to-bottom independently.

In [ ]:
from __future__ import annotations

import inspect
import json
import re
from pathlib import Path
from typing import Any, Awaitable, Callable

from utils import DocumentType, penrs_fetch_document

## Type Aliases & Constants

Callable type aliases for the three injectable dependencies (`rubric_fetcher`, `document_fetcher`, `llm_invoker`) and module-level constants for the default rubric path and truncation marker template.

In [ ]:
RubricFetcher = Callable[[str], Any | Awaitable[Any]]
DocumentFetcher = Callable[[str, DocumentType, dict[str, str] | None], Any | Awaitable[Any]]
LLMInvoker = Callable[..., Any | Awaitable[Any]]

_DEFAULT_RUBRICS_PATH = Path("rubrics.json")
_JSON_ONLY_SYSTEM_PROMPT = "Respond only in valid JSON."
_TRUNCATION_MARKER_TEMPLATE = "\n...[truncated {count} chars]...\n"


def _callable_accepts_kwarg(fn: Callable[..., Any], kwarg_name: str) -> bool:
    try:
        sig = inspect.signature(fn)
    except (TypeError, ValueError):
        return False

    for param in sig.parameters.values():
        if param.kind == inspect.Parameter.VAR_KEYWORD:
            return True
        if param.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY):
            if param.name == kwarg_name:
                return True
    return False

## Helper Utilities

Internal helpers used by `PENRSWorker.run()` and `PENRSWorker.parse_json_response()`: an async-compatibility shim (`_maybe_await`), a rubric loader from disk (`_load_rubric_from_json`), a type-safe text coercer (`_coerce_text`), and a best-effort JSON extractor from prose text (`_extract_json_from_text`).

In [ ]:
async def _maybe_await(value: Any) -> Any:
    if hasattr(value, "__await__"):
        return await value
    return value


def _load_rubric_from_json(rubric_id: str) -> dict[str, Any]:
    if not _DEFAULT_RUBRICS_PATH.exists():
        return {"rubric_id": rubric_id, "error": "rubrics.json not found"}

    try:
        payload = json.loads(_DEFAULT_RUBRICS_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        return {"rubric_id": rubric_id, "error": f"Unable to read rubrics.json: {exc}"}

    if isinstance(payload, dict):
        rubric = payload.get(rubric_id)
        if isinstance(rubric, dict):
            return rubric
        return {"rubric_id": rubric_id, "error": f"Rubric '{rubric_id}' not found"}

    return {"rubric_id": rubric_id, "error": "rubrics.json must contain an object"}


def _coerce_text(document_payload: Any) -> str:
    if isinstance(document_payload, str):
        return document_payload
    if isinstance(document_payload, bytes):
        return document_payload.decode("utf-8", errors="replace")
    if document_payload is None:
        return ""
    if isinstance(document_payload, (dict, list, tuple)):
        return json.dumps(document_payload, ensure_ascii=True, sort_keys=True)
    return str(document_payload)


def _extract_json_from_text(text: str) -> Any | None:
    decoder = json.JSONDecoder()
    for match in re.finditer(r"[{\[]", text):
        start = match.start()
        try:
            parsed, _end = decoder.raw_decode(text[start:])
            return parsed
        except json.JSONDecodeError:
            continue
    return None

## Truncation

`truncate_for_context` keeps the head and tail of a document excerpt when it exceeds the LLM context budget (`max_chars`). A marker is inserted in the middle reporting how many characters were dropped, so the LLM always sees both the beginning and end of the document.

In [ ]:
def truncate_for_context(text: str, max_chars: int = 12000) -> str:
    if max_chars <= 0:
        raise ValueError("max_chars must be greater than zero")
    if len(text) <= max_chars:
        return text

    truncated_count = len(text) - max_chars
    marker = _TRUNCATION_MARKER_TEMPLATE.format(count=truncated_count)
    if len(marker) >= max_chars:
        return text[:max_chars]

    remaining = max_chars - len(marker)
    head_len = remaining // 2
    tail_len = remaining - head_len
    return f"{text[:head_len]}{marker}{text[-tail_len:]}"

## PENRSWorker Class

The `PENRSWorker` base class wires together rubric fetching, document routing, LLM invocation, and JSON parsing into a single `run()` coroutine. All three injectable dependencies (`rubric_fetcher`, `document_fetcher`, `llm_invoker`) have sensible defaults and can be overridden for testing or to plug in real API/LLM clients.

In [ ]:
class PENRSWorker:
    def __init__(
        self,
        *,
        name: str,
        weight: float,
        signal_density: float,
        rubric_id: str,
        document_type: DocumentType,
        rubric_fetcher: RubricFetcher | None = None,
        document_fetcher: DocumentFetcher | None = None,
        llm_invoker: LLMInvoker | None = None,
        max_context_chars: int = 12000,
    ) -> None:
        self.name = name
        self.weight = weight
        self.signal_density = signal_density
        self.rubric_id = rubric_id
        self.document_type = document_type
        self.rubric_fetcher = rubric_fetcher or _load_rubric_from_json
        self.document_fetcher = document_fetcher or penrs_fetch_document
        self.llm_invoker = llm_invoker or (lambda _prompt, *, system=_JSON_ONLY_SYSTEM_PROMPT: "{}")
        self._llm_accepts_system = _callable_accepts_kwarg(self.llm_invoker, "system")
        self.max_context_chars = max_context_chars

    @property
    def metadata(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "weight": self.weight,
            "signal_density": self.signal_density,
        }

    def parse_json_response(self, response: Any) -> dict[str, Any]:
        default_payload = {"score": 0.0, "thesis": "Parse failure", "evidence_nodes": []}

        def _enforce_schema(payload: Any) -> dict[str, Any]:
            if not isinstance(payload, dict):
                return default_payload.copy()

            required = ("score", "thesis", "evidence_nodes")
            if any(key not in payload for key in required):
                return default_payload.copy()

            try:
                score = float(payload["score"])
            except (TypeError, ValueError):
                score = default_payload["score"]
            score = max(-1.0, min(1.0, score))

            thesis = payload["thesis"] if isinstance(payload["thesis"], str) else default_payload["thesis"]
            evidence_nodes = payload["evidence_nodes"] if isinstance(payload["evidence_nodes"], list) else []

            return {
                "score": score,
                "thesis": thesis,
                "evidence_nodes": evidence_nodes,
            }

        if isinstance(response, dict):
            return _enforce_schema(response)
        if response is None:
            return default_payload.copy()

        text = str(response).strip()
        if not text:
            return default_payload.copy()

        try:
            parsed = json.loads(text)
            return _enforce_schema(parsed)
        except json.JSONDecodeError:
            pass

        fenced_match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, flags=re.IGNORECASE)
        if fenced_match:
            fenced_payload = fenced_match.group(1).strip()
            try:
                parsed = json.loads(fenced_payload)
                return _enforce_schema(parsed)
            except json.JSONDecodeError:
                pass

        extracted = _extract_json_from_text(text)
        if extracted is not None:
            return _enforce_schema(extracted)

        return default_payload.copy()

    def build_prompt(
        self,
        *,
        ticker: str,
        date_from: str,
        date_to: str,
        rubric: dict[str, Any],
        document_excerpt: str,
    ) -> str:
        return (
            f"Worker: {self.name}\n"
            f"Ticker: {ticker}\n"
            f"Date range: {date_from} -> {date_to}\n"
            f"Rubric JSON: {json.dumps(rubric, ensure_ascii=True)}\n"
            f"Document excerpt:\n{document_excerpt}\n"
            "Return only JSON.\n"
            "You MUST return a strictly valid JSON object adhering to this exact schema:\n"
            "{\n"
            '  "score": <float between -1.0 and 1.0>,\n'
            '  "thesis": <string>,\n'
            '  "evidence_nodes": [\n'
            '    {"verbatim_quote": <exact substring from document>, "reasoning": <string>}\n'
            "  ]\n"
            "}\n"
            "The 'verbatim_quote' MUST be an exact, character-for-character substring of the provided Document excerpt."
        )

    async def run(self, ticker: str, date_from: str, date_to: str) -> dict[str, Any]:
        rubric_raw = await _maybe_await(self.rubric_fetcher(self.rubric_id))
        rubric = rubric_raw if isinstance(rubric_raw, dict) else {"rubric": rubric_raw}

        doc_result_raw = await _maybe_await(
            self.document_fetcher(
                ticker,
                self.document_type,
                {"from": date_from, "to": date_to},
            )
        )
        doc_result = (
            doc_result_raw
            if isinstance(doc_result_raw, dict)
            else {"status": "not_released", "data": {}}
        )

        if doc_result.get("status") == "not_released":
            data = doc_result.get("data", {})
            return {
                "status": "not_released",
                "worker": self.metadata,
                "ticker": ticker,
                "date_from": date_from,
                "date_to": date_to,
                "document_type": self.document_type.value,
                "apis_attempted": list(data.get("apis_attempted", [])),
                "detail": data,
            }

        document_data = doc_result.get("data", {})
        excerpt = truncate_for_context(_coerce_text(document_data), max_chars=self.max_context_chars)
        prompt = self.build_prompt(
            ticker=ticker,
            date_from=date_from,
            date_to=date_to,
            rubric=rubric,
            document_excerpt=excerpt,
        )

        if self._llm_accepts_system:
            llm_call = self.llm_invoker(prompt, system=_JSON_ONLY_SYSTEM_PROMPT)
        else:
            llm_call = self.llm_invoker(prompt)

        llm_raw = await _maybe_await(llm_call)
        parsed = self.parse_json_response(llm_raw)

        original_node_count = len(parsed.get("evidence_nodes", []))
        validated_nodes: list[dict[str, Any]] = []
        for node in parsed.get("evidence_nodes", []):
            if not isinstance(node, dict):
                continue
            quote = node.get("verbatim_quote")
            if isinstance(quote, str) and quote in excerpt:
                validated_nodes.append(node)

        parsed["evidence_nodes"] = validated_nodes

        # Neutralize if nodes were provided but all hallucinated, or if score is
        # high-conviction (|score| >= 0.5) but the LLM supplied no evidence at all.
        all_nodes_hallucinated = original_node_count > 0 and len(validated_nodes) == 0
        high_conviction_no_evidence = original_node_count == 0 and abs(parsed["score"]) >= 0.5
        if all_nodes_hallucinated or high_conviction_no_evidence:
            parsed["score"] = 0.0
            note = "[SYSTEM NOTE: Score neutralized due to hallucinated evidence.]"
            thesis = parsed.get("thesis", "")
            thesis_text = thesis if isinstance(thesis, str) else str(thesis)
            if note not in thesis_text:
                parsed["thesis"] = f"{thesis_text.rstrip()} {note}".strip()

        return {
            "status": "available",
            "worker": self.metadata,
            "ticker": ticker,
            "date_from": date_from,
            "date_to": date_to,
            "document_type": self.document_type.value,
            "rubric": rubric,
            "result": parsed,
        }